# Phenotype permutation test for FastLMM GWAS

This notebook generates **100 permuted phenotype columns** by shuffling phenotype values among strains and then runs a FastLMM genome-wide association study (GWAS) on all permutations. The genotype data and strain identifiers remain unchanged.

The permutation results provide an empirical null distribution of p-values. In this project, the final p-value threshold is selected manually in Excel by retaining the lowest 5% of permutation p-values and choosing the highest p-value within that subset.

> The notebook creates the permutation results but does not automatically select the final threshold.

## 1. Imports and configuration

Update the paths in this cell for the phenotype being tested. The phenotype file must be whitespace-delimited and contain at least three columns: family ID (`FID`), individual ID (`IID`), and phenotype value. It should not contain a header row.

The PLINK genotype prefix identifies matching `.bed`, `.bim`, and `.fam` files. Missing phenotype values coded as `-9` remain attached to their original strains and are not included in the shuffle. A fixed random seed makes the 100 permutations reproducible.

In [ ]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd
from fastlmm.association import single_snp

logging.basicConfig(level=logging.WARNING)
pd.set_option("display.width", 1000)

# Analysis settings
N_PERMUTATIONS = 100
RANDOM_SEED = 12345
MISSING_PHENOTYPE_VALUE = -9
COUNT_A1 = False

# Input files
BED_PREFIX = Path("path/to/plink_dataset")
PHENOTYPE_FILE = Path(
    "path/to/phenotype.txt"
)
COVARIATE_FILE = Path("path/to/covariates.txt")

# Replace this with the folder where generated files should be written.
OUTPUT_DIRECTORY = Path("path/to/output")
PERMUTED_PHENOTYPE_FILE = OUTPUT_DIRECTORY / (
    f"{PHENOTYPE_FILE.stem}_permutation_{N_PERMUTATIONS}_times.txt"
)
GWAS_RESULTS_FILE = OUTPUT_DIRECTORY / (
    f"{PHENOTYPE_FILE.stem}_post_permutation.csv"
)

## 2. Validate and load the phenotype data

This section checks that all required inputs exist, loads the first three phenotype-file columns, verifies that phenotype values are numeric, and confirms that each `FID`/`IID` pair is unique. These checks prevent a long GWAS run from starting with malformed input.

In [ ]:
required_files = [
    Path(f"{BED_PREFIX}.bed"),
    Path(f"{BED_PREFIX}.bim"),
    Path(f"{BED_PREFIX}.fam"),
    PHENOTYPE_FILE,
    COVARIATE_FILE,
]
missing_files = [path for path in required_files if not path.is_file()]
if missing_files:
    missing_list = "\n".join(f"  - {path}" for path in missing_files)
    raise FileNotFoundError(f"Required input file(s) not found:\n{missing_list}")

phenotype_table = pd.read_csv(
    PHENOTYPE_FILE,
    sep=r"\s+",
    header=None,
    dtype=str,
)
if phenotype_table.empty:
    raise ValueError(f"The phenotype file contains no rows: {PHENOTYPE_FILE}")
if phenotype_table.shape[1] < 3:
    raise ValueError(
        "The phenotype file must contain FID, IID, and phenotype columns."
    )

phenotype_table = phenotype_table.iloc[:, :3].copy()
phenotype_table.columns = ["FID", "IID", "Phenotype"]
phenotype_table["Phenotype"] = pd.to_numeric(
    phenotype_table["Phenotype"], errors="coerce"
)

invalid_rows = phenotype_table.index[phenotype_table["Phenotype"].isna()].tolist()
if invalid_rows:
    raise ValueError(
        "Non-numeric or missing phenotype values were found at zero-based row "
        f"indices: {invalid_rows[:10]}"
    )
if phenotype_table[["FID", "IID"]].duplicated().any():
    raise ValueError("Duplicate FID/IID pairs were found in the phenotype file.")

OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
observed_count = (
    phenotype_table["Phenotype"] != MISSING_PHENOTYPE_VALUE
).sum()
missing_count = len(phenotype_table) - observed_count

print(f"Strains loaded: {len(phenotype_table):,}")
print(f"Observed phenotypes: {observed_count:,}")
print(f"Missing phenotypes ({MISSING_PHENOTYPE_VALUE}): {missing_count:,}")

## 3. Generate 100 shuffled phenotypes

For each permutation, the observed phenotype values are randomly reassigned among strains. The `FID` and `IID` columns remain fixed, and missing-value positions remain fixed. Each shuffled phenotype becomes one column in the output file.

The output contains exactly one row per input strain; the earlier version of this notebook wrote the first row twice.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
phenotype_values = phenotype_table["Phenotype"].to_numpy()
observed_mask = phenotype_values != MISSING_PHENOTYPE_VALUE
observed_values = phenotype_values[observed_mask]

permuted_columns = {}
for permutation_number in range(1, N_PERMUTATIONS + 1):
    permuted_values = phenotype_values.copy()
    permuted_values[observed_mask] = rng.permutation(observed_values)
    permuted_columns[f"Permutation_{permutation_number:03d}"] = permuted_values

permuted_table = pd.concat(
    [
        phenotype_table[["FID", "IID"]].reset_index(drop=True),
        pd.DataFrame(permuted_columns),
    ],
    axis=1,
)
permuted_table.to_csv(
    PERMUTED_PHENOTYPE_FILE,
    sep="\t",
    header=False,
    index=False,
)

assert len(permuted_table) == len(phenotype_table)
assert permuted_table.shape[1] == N_PERMUTATIONS + 2
print(
    f"Saved {N_PERMUTATIONS} permutations for {len(permuted_table):,} strains to:"
)
print(PERMUTED_PHENOTYPE_FILE)
permuted_table.head()

## 4. Run FastLMM on the permuted phenotypes

FastLMM runs a single-SNP association test for the 100 shuffled phenotype columns using the same PLINK genotype data and covariates. Depending on the dataset size, this step may take a long time. The full results are saved as a CSV for the manual threshold calculation.

In [ ]:
results_df = single_snp(
    test_snps=str(BED_PREFIX),
    pheno=str(PERMUTED_PHENOTYPE_FILE),
    covar=str(COVARIATE_FILE),
    count_A1=COUNT_A1,
)

if "PValue" not in results_df.columns:
    raise KeyError(
        f"FastLMM results do not contain a PValue column: {list(results_df.columns)}"
    )

results_df.to_csv(GWAS_RESULTS_FILE, index=False)
print(f"GWAS result rows: {len(results_df):,}")
print(f"Full permutation results saved to: {GWAS_RESULTS_FILE}")
results_df.head()

## 5. Select the p-value threshold manually in Excel

The final threshold used in this project is selected manually from the permutation GWAS results:

1. Open the CSV saved at `GWAS_RESULTS_FILE` in Excel.
2. Identify the `PValue` column containing the p-values from the 100 shuffled-phenotype GWAS runs.
3. Sort the p-values from smallest to largest, or apply a filter that retains the **lowest 5%** of all p-values.
4. Within that lowest-5% subset, select the **highest p-value**.
5. Record that value as the permutation-derived p-value threshold used to filter the real GWAS results.